In [1]:
import torch
from sympy import sequence

from torch import nn

from transformers import AutoTokenizer
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast
from transformers.processing_utils import Unpack
from transformers.utils import TransformersKwargs, auto_docstring, can_return_tuple
from transformers.models.qwen3 import Qwen3PreTrainedModel, Qwen3Model, Qwen3ForCausalLM
from transformers.cache_utils import Cache
from transformers.generation.utils import GenerationMixinForTemplatedToolCalling

from typing import Optional, Union, List


In [2]:
# Config
MODEL_NAME = "Qwen/Qwen3-8B-AWQ"
USE_THINKING=False


In [3]:
class CustomQwen3ForCausalLM(Qwen3PreTrainedModel, GenerationMixinForTemplatedToolCalling):

    _tied_weights_keys = ["lm_head.weight"]
    _tp_plan = {"lm_head": "colwise_rep"}
    _pp_plan = {"lm_head": (["hidden_states"], ["logits"])}

    def __init__(self, config):
        super().__init__(config)
        self.model = Qwen3Model(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Initialize weights and apply final processing
        self.post_init()

        # Placeholder for tool calling tags
        self.special_tag_registry = {}

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Cache] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        logits_to_keep: Union[int, torch.Tensor] = 0,
        **kwargs: Unpack[TransformersKwargs],
    ) -> CausalLMOutputWithPast:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
            config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored
            (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.

        Example:

        ```python
        >>> from transformers import AutoTokenizer, Qwen3ForCausalLM

        >>> model = Qwen3ForCausalLM.from_pretrained("Qwen/Qwen3-8B")
        >>> tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

        >>> prompt = "Hey, are you conscious? Can you talk to me?"
        >>> inputs = tokenizer(prompt, return_tensors="pt")

        >>> # Generate
        >>> generate_ids = model.generate(inputs.input_ids, max_length=30)
        >>> tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        "Hey, are you conscious? Can you talk to me?\nI'm not conscious, but I can talk to you."
        ```"""
        outputs: BaseModelOutputWithPast = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            cache_position=cache_position,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        # Only compute necessary logits, and do not upcast them to float if we are not computing the loss
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
        logits = self.lm_head(hidden_states[:, slice_indices, :])

        loss = None
        if labels is not None:
            loss = self.loss_function(logits=logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    def preprocess_input_seq_before_generation_step(self, input_ids, model_kwargs, step_counter):
        batch_size = input_ids.size(0)

        iterator = 0
        for batch_idx in range(batch_size):
            sequence = input_ids[batch_idx]
            # 198 is a newline character.
            has_think_tokens = self._find_token_in_sequence(sequence, target_token=[self.special_tag_registry["think"]["start"], 198, self.special_tag_registry["think"]["end"]])

            # At this point we need to make sure, we only catch the actual tool calls and not chat template artifacts.
            if has_think_tokens:
                if sequence[-1] == self.special_tag_registry["tool_call"]["start"]:
                    print(f"Started tool call. Token: {sequence[-1]}. Step: {step_counter}. Iteration: {iterator}")
                elif sequence[-1] == self.special_tag_registry["tool_call"]["end"]:
                    print(f"Ended tool call. Token: {sequence[-1]}. Previous token: {sequence[-2]}. Step: {step_counter}. Iteration: {iterator}")

            iterator += 1

        return input_ids, model_kwargs

    def encode_special_tags(self, tag_key: str, tags: List[str], auto_tokenizer: AutoTokenizer) -> None:

        self.special_tag_registry.update({tag_key: {
            "start": auto_tokenizer.encode(tags[0], return_tensors="pt").squeeze().item(),
            "end": auto_tokenizer.encode(tags[1], return_tensors="pt").squeeze().item() if len(tags) > 1 else None,
        }})

        print(f"Encoded special tokens: {self.special_tag_registry[tag_key]}")

    @staticmethod
    def _find_token_in_sequence(input_ids: torch.Tensor, target_token: Union[torch.Tensor, int, List]) -> bool:

        if type(target_token) is list:
            target_token = torch.tensor(target_token, device=input_ids.device)
        elif isinstance(target_token, int):
            target_token = torch.tensor([target_token], device=input_ids.device)

        if len(input_ids) == 0 or len(target_token) == 0:
            return False

        if len(target_token) == 1:
            # Solution for single-token search
            return torch.any(torch.isin(input_ids, target_token))

        # For sequences, we need to find the exact subsequence
        target_len = len(target_token)
        input_len = len(input_ids)

        if target_len > input_len:
            return False

        # Vectorized subsequence search using unfold
        # Creates a sliding window view of input_ids
        windows = input_ids.unfold(0, target_len, 1)  # Shape: (num_windows, target_len)

        # Compare each window with target sequence
        matches = torch.all(windows == target_token.unsqueeze(0), dim=1)

        return torch.any(matches).item()


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
	MODEL_NAME,
	enable_thinking=USE_THINKING,
)
model = CustomQwen3ForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda"
)

model.encode_special_tags(tag_key="tool_call", tags=["<tool_call>", "</tool_call>"], auto_tokenizer=tokenizer)
model.encode_special_tags(tag_key="think",tags=["<think>", "</think>"], auto_tokenizer=tokenizer)


/home/herbert/code/transformers/.venv/lib/python3.12/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Encoded special tokens: {'start': 151657, 'end': 151658}
Encoded special tokens: {'start': 151667, 'end': 151668}


In [5]:
messages = [
    {"role": "system", "content": "You are an expert in tool calling and do exactly as the user tells you. DO NOT THINK!"},
    {"role": "user", "content": "Can you please calculate the sum of 10 + 2 + 4 and then multiply the result by 10? This requires you to call the calculate() tool twice."},
]
tools = [{
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Calculate the result of a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to calculate, such as '2 + 2'. The expression can contain numbers, operators (+, -, *, /), parentheses, and spaces.",
                },
            },
            "required": ["expression"],
        },
    },
}]

# Set pad token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Generate with various parameters
# print(hasattr(model, '_extract_generation_mode_kwargs'))
# print(model.__class__.__mro__)  # Check method resolution order
with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_length=500,
        num_return_sequences=1,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        top_p=0.9,
        # repetition_penalty=1.1
    )

# Decode and print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(outputs)
print(generated_text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Started tool call. Token: 151657. Step: 5. Iteration: 0
Started tool call. Token: 151657. Step: 5. Iteration: 0
Ended tool call. Token: 151658. Previous token: 95642. Step: 29. Iteration: 0
Ended tool call. Token: 151658. Previous token: 95642. Step: 29. Iteration: 0
Started tool call. Token: 151657. Step: 31. Iteration: 0
Started tool call. Token: 151657. Step: 31. Iteration: 0
Ended tool call. Token: 151658. Previous token: 95642. Step: 53. Iteration: 0
Ended tool call. Token: 151658. Previous token: 95642. Step: 53. Iteration: 0
tensor([[151644,   8948,    198,   2610,    525,    458,   6203,    304,   5392,
           8098,    323,    653,   6896,    438,    279,   1196,  10742,    498,
             13,   9319,   4183,  92119,   2219,      2,  13852,    271,   2610,
           1231,   1618,    825,    476,    803,   5746,    311,   7789,    448,
            279,   1196,   3239,    382,   2610,    525,   3897,    448,    729,
          32628,   2878,    366,  15918,   1472,  15918, 

In [6]:
tns = torch.tensor([[151644,   8948,    198,   2610,    525,    458,   6203,    304,   5392,
           8098,    323,    653,   6896,    438,    279,   1196,  10742,    498,
             13,   9319,   4183,  92119,   2219,      2,  13852,    271,   2610,
           1231,   1618,    825,    476,    803,   5746,    311,   7789,    448,
            279,   1196,   3239,    382,   2610,    525,   3897,    448,    729,
          32628,   2878,    366,  15918,   1472,  15918,     29,  11874,   9492,
            510,     27,  15918,    397,   4913,   1313,    788,    330,   1688,
            497,    330,   1688,    788,   5212,    606,    788,    330,  35597,
            497,    330,   4684,    788,    330,  47866,    279,   1102,    315,
            264,  35972,   7493,  10465,    330,  13786,    788,   5212,   1313,
            788,    330,   1700,    497,    330,  13193,    788,   5212,  28099,
            788,   5212,   1313,    788,    330,    917,    497,    330,   4684,
            788,    330,    785,  35972,   7493,    311,  11047,     11,   1741,
            438,    364,     17,    488,    220,     17,   4427,    576,   7493,
            646,   6644,   5109,     11,  19624,  17973,     11,  85922,  11777,
            608,    701,  73975,     11,    323,  12621,   1189,  38154,    330,
           6279,    788,   4383,  28099,   1341,   3417,    532,    522,  15918,
           1339,   2461,   1817,    729,   1618,     11,    470,    264,   2951,
           1633,    448,    729,    829,    323,   5977,   2878,    220, 151657,
         151658,  11874,   9492,    510, 151657,    198,   4913,    606,    788,
            366,   1688,  11494,   8066,    330,  16370,    788,    366,   2116,
          56080,  40432,  31296, 151658, 151645,    198, 151644,    872,    198,
           6713,    498,   4486,  11047,    279,   2629,    315,    220,     16,
             15,    488,    220,     17,    488,    220,     19,    323,   1221,
          30270,    279,   1102,    553,    220,     16,     15,     30,   1096,
           7460,    498,    311,   1618,    279,  11047,    368,   5392,  10917,
             13, 151645,    198, 151644,  77091,    198, 151667,    198, 151668,
            271, 151657,    198,   4913,    606,    788,    330,  35597,    497,
            330,  16370,    788,   5212,  28099,    788,    330,     16,     15,
            488,    220,     17,    488,    220,     19,  95642, 151658,    198,
         151657,    198,   4913,    606,    788,    330,  35597,    497,    330,
          16370,    788,   5212,  28099,    788,    330,  19702,   5287,    353,
            220,     16,     15,  95642, 151658, 151645]])

def find_token_in_sequence(input_ids: torch.Tensor, target_token: Union[torch.Tensor, int, List]) -> bool:

        if type(target_token) is list:
            target_token = torch.tensor(target_token, device=input_ids.device)

        if len(input_ids) == 0:
            return False

        has_token = torch.any(torch.isin(input_ids, target_token))
        return has_token


display(find_token_in_sequence(input_ids=tns, target_token=torch.tensor([151667, 151668])))

tensor(True)